# Barcelona Noise Prediction: Lidar Building Heights
This notebook downloads the building footprints from OpenStreetMap (OSM) and extracts highly precise height data by evaluating zonal statistics over two high-resolution LiDAR Digital Surface Model TIFFs (`Catalunya-1mtif1777965317095.tif` and `Catalunya-1mtif1777965631099.tif`).
We then aggregate these heights within street-level buffers.

In [ ]:
import geopandas as gpd
import pandas as pd
import osmnx as ox
import matplotlib.pyplot as plt
import os
import numpy as np
import rasterio
from rasterio.merge import merge
from rasterstats import zonal_stats

ox.settings.timeout = 1000

print("Loading street data...")
noise_streets = gpd.read_file("../layers/BCN_noise_streets.gpkg")
print("Streets CRS:", noise_streets.crs)
print("Number of street segments:", len(noise_streets))

## 1. Download Building Data from OSM
We need the footprints to act as our 'cookie cutters' against the LiDAR rasters.

In [ ]:
place_name = "Barcelona, Spain"
tags = {'building': True}

print("Downloading building footprints...")
try:
    buildings = ox.features_from_place(place_name, tags)
except AttributeError:
    buildings = ox.geometries_from_place(place_name, tags)

buildings = buildings[buildings.geometry.type.isin(['Polygon', 'MultiPolygon'])]
buildings = buildings.to_crs(noise_streets.crs)
print(f"Loaded {len(buildings)} building footprints.")

## 2. Load and Mosaic the LiDAR TIFFs
We will merge the two raster datasets together into a single continuous raster memory object so we can evaluate buildings across the boundary line seamlessly.

In [ ]:
tif_paths = [
    "../layers/Digital_terrain_model/superficies-correlacio-Catalunya/Catalunya-1mtif1777965317095.tif",
    "../layers/Digital_terrain_model/superficies-correlacio-Catalunya/Catalunya-1mtif1777965631099.tif"
]

src_files_to_mosaic = []
for fp in tif_paths:
    src = rasterio.open(fp)
    src_files_to_mosaic.append(src)

print("Merging TIFFs (creating mosaic)...")
mosaic, out_trans = merge(src_files_to_mosaic)

# Check CRS 
raster_crs = src_files_to_mosaic[0].crs
print(f"Raster CRS is: {raster_crs}")

if buildings.crs != raster_crs:
    print("Aligning Building footprints explicitly to Raster CRS...")
    buildings = buildings.to_crs(raster_crs)

# Temporarily save the mosaic to disk so rasterstats can compute efficiently in chunks
output_mosaic_path = "../data/processed/merged_lidar.tif"
os.makedirs("../data/processed", exist_ok=True)

out_meta = src_files_to_mosaic[0].meta.copy()
out_meta.update({
    "driver": "GTiff",
    "height": mosaic.shape[1],
    "width": mosaic.shape[2],
    "transform": out_trans
})

print("Saving mosaiced raster temporarily to disk...")
with rasterio.open(output_mosaic_path, "w", **out_meta) as dest:
    dest.write(mosaic)

# Close readers
for src in src_files_to_mosaic:
    src.close()

## 3. Extract Zonal Statistics
We drape our building polygons over the mosaiced LiDAR raster and calculate the max/min elevations to extract the 'range' which gives us the actual building height (Difference between roof height and ground level touching the polygon).

In [ ]:
print("Calculating height statistics for each building (zonal_stats). This may take a while...")
# In the case of Surface Models, max-min=range provides the most stable height value if it's absolute AMSL.
# If the raster is a normalized DSM (nDSM) where ground=0, then 'max' or 'mean' is the height.
stats = zonal_stats(buildings, output_mosaic_path, stats="min mean max range", geojson_out=False)

stats_df = pd.DataFrame(stats)

# If using absolute Digital Surface Model, the range (max - min) within the footprint is a good proxy for building height
# We also keep the mean and max absolute elevation if needed, but height is standard distance from ground.
buildings['lidar_estimated_height'] = stats_df['range'].fillna(0)
buildings['lidar_max_elev'] = stats_df['max'].fillna(0)
buildings['lidar_mean_elev'] = stats_df['mean'].fillna(0)

display(buildings[['geometry', 'lidar_estimated_height', 'lidar_max_elev', 'lidar_mean_elev']].head(5))

buildings['lidar_estimated_height'].plot(kind='hist', bins=50, figsize=(6, 4), color='orange', edgecolor='black')
plt.title('Distribution of LiDAR Building Heights (Range)')
plt.xlim(0, 50)
plt.show()

## 4. Buffer Analysis (Street Matching)
Just like before, we compute the average and maximum building heights enclosed closely by the 50m and 100m street buffers.

In [ ]:
# Before SJOIN, let's make sure streets and buildings share the exact same CRS
if noise_streets.crs != buildings.crs:
    noise_streets = noise_streets.to_crs(buildings.crs)

def calculate_lidar_heights_in_buffer(streets_gdf, buildings_gdf, buffer_size):
    print(f"Evaluating {buffer_size}m buffers...")
    buffered_streets = streets_gdf.copy()
    buffered_streets['geometry'] = buffered_streets.geometry.buffer(buffer_size)
    
    bldgs = buildings_gdf[['lidar_estimated_height', 'geometry']].copy()
    joined = gpd.sjoin(buffered_streets[['TRAM', 'geometry']], bldgs, how='left', predicate='intersects')
    joined['lidar_estimated_height'] = joined['lidar_estimated_height'].fillna(0)
    
    agg_funcs = {
        'lidar_estimated_height': ['mean', 'max']
    }
    grouped = joined.groupby('TRAM').agg(agg_funcs)
    grouped.columns = [f"lidar_h_{col[1]}_{buffer_size}m" for col in grouped.columns]
    
    return grouped.reset_index()

features_50m = calculate_lidar_heights_in_buffer(noise_streets, buildings, 50)
features_100m = calculate_lidar_heights_in_buffer(noise_streets, buildings, 100)

display(features_50m.head())

## 5. Merge, Export and Visualize

In [ ]:
dataset = pd.DataFrame({'street_id': noise_streets['TRAM']})

dataset = dataset.merge(features_50m, left_on='street_id', right_on='TRAM', how='left').drop(columns=['TRAM'])
dataset = dataset.merge(features_100m, left_on='street_id', right_on='TRAM', how='left').drop(columns=['TRAM'])
dataset = dataset.fillna(0)

output_dir = "../data/processed"
dataset.to_csv(os.path.join(output_dir, "lidar_building_heights.csv"), index=False)
print("Exported lidar_building_heights.csv")

# Visualization
viz_gdf = noise_streets.merge(dataset[['street_id', 'lidar_h_mean_50m']], left_on='TRAM', right_on='street_id')
vmax_height = viz_gdf['lidar_h_mean_50m'].quantile(0.95)

fig, ax = plt.subplots(figsize=(12, 10))
viz_gdf.plot(
    column='lidar_h_mean_50m',
    ax=ax,
    cmap='magma',
    linewidth=1.5,
    legend=True,
    legend_kwds={'label': "Average LiDAR Building Height in 50m Buffer (m)"},
    vmax=vmax_height
)
ax.set_title("Barcelona Street Network: Average Local LiDAR Building Height", fontsize=16)
ax.set_axis_off()
plt.show()